# 07 - MACE-MP-0 smoke + extracao de features

Etapa 3, Tarefa 2. Confirma MACE-MP-0 na GPU, inspeciona a API real do mace-torch instalado, e valida os `.npz` produzidos por `scripts/07_extract_mace_features.py`.

Logica em `src/her_gnn/models/mace_features.py`; este notebook so orquestra e valida.

In [1]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
assert torch.cuda.is_available(), "CUDA indisponivel"
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: NVIDIA GeForce RTX 5060 Ti


## Passo 1 - Smoke test (estrutura sintetica)

In [2]:
from ase.build import bulk
from mace.calculators import mace_mp

calc = mace_mp(model="medium", device="cuda", default_dtype="float32")
atoms = bulk("Cu", "fcc", a=3.6)
atoms.calc = calc
print(f"E(Cu bulk) = {atoms.get_potential_energy():.3f} eV")
print(f"VRAM pico: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

/home/mauricio/repositorios/her-gnn/.venv/lib/python3.14/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


Using Materials Project MACE for MACECalculator with /home/mauricio/.cache/mace/20231203mace128L1_epoch199model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/mauricio/repositorios/her-gnn/.venv/lib/python3.14/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


E(Cu bulk) = -4.082 eV
VRAM pico: 0.10 GB


## Passo 2 - API real do mace-torch

Numa estrutura HER real, conferir o que `calc.results` expoe nesta versao.

In [3]:
from ase.io import read

a = read("data/processed/her_dataset.traj", index=0)
a.calc = calc
E = a.get_potential_energy()
res = a.calc.results
print("formula:", a.get_chemical_formula())
print("results keys:", list(res.keys()))
print("tem 'energies' (per-atom):", "energies" in res, np.asarray(res["energies"]).shape)
print("soma per-atom == E_total:", np.isclose(np.asarray(res["energies"]).sum(), E, atol=1e-2))
print("\nVersao instalada expoe energias per-atom -> 7 features escalares OK.")
print("Embeddings de no exigem forward manual via calc.models[0]; adiados (escalares bastam para A.5).")

formula: HOs12
results keys: ['energy', 'node_energy', 'forces', 'stress', 'free_energy', 'energies']
tem 'energies' (per-atom): True (13,)
soma per-atom == E_total: True

Versao instalada expoe energias per-atom -> 7 features escalares OK.
Embeddings de no exigem forward manual via calc.models[0]; adiados (escalares bastam para A.5).


## Passo 5 - Validacao pos-extracao

Rode antes: `uv run python scripts/07_extract_mace_features.py`. As celulas abaixo validam os `.npz`.

In [4]:
MACE_DIR = Path("data/mace_features")
ready = (MACE_DIR / "train.npz").exists()
if not ready:
    print("data/mace_features/train.npz ausente — rode scripts/07_extract_mace_features.py primeiro.")
else:
    splits_json = json.load(open("data/splits.json"))
    total = 0
    names_ref = None
    for name in ["train", "val", "test"]:
        d = np.load(MACE_DIR / f"{name}.npz", allow_pickle=True)
        total += len(d["ids"])
        names_ref = names_ref or list(d["feature_names"])
        assert list(d["feature_names"]) == names_ref, "feature_names inconsistente"
        assert not np.isnan(d["X"]).any(), f"NaN em {name}"
        assert not np.isinf(d["X"]).any(), f"Inf em {name}"
        assert d["y"].min() >= -2.1 and d["y"].max() <= 2.1, f"y fora do range em {name}"
        assert d["X"][:, 0].mean() < 0, "E_total deveria ser negativa"
        print(f"{name}: {len(d['ids'])} estruturas, {d['X'].shape[1]} features")
    assert total == 5860, f"total {total} != 5860"
    train_ids = set(np.load(MACE_DIR / 'train.npz', allow_pickle=True)['ids'].tolist())
    val_ids = set(np.load(MACE_DIR / 'val.npz', allow_pickle=True)['ids'].tolist())
    test_ids = set(np.load(MACE_DIR / 'test.npz', allow_pickle=True)['ids'].tolist())
    assert train_ids | val_ids == set(splits_json["train"]), "train+val != splits['train']"
    assert test_ids == set(splits_json["test"]), "test != splits['test']"
    print("\nOK: total=5860, ids alinhados ao split canonico, sem NaN/Inf.")

data/mace_features/train.npz ausente — rode scripts/07_extract_mace_features.py primeiro.


In [5]:
if ready:
    d = np.load(MACE_DIR / "train.npz", allow_pickle=True)
    names = list(d["feature_names"])
    fig, axes = plt.subplots(3, 3, figsize=(12, 9))
    for ax, name, col in zip(axes.flat, names, d["X"].T):
        ax.hist(col, bins=50, color="#4c72b0")
        ax.set_title(name, fontsize=9)
    for ax in axes.flat[len(names):]:
        ax.set_axis_off()
    fig.tight_layout()
    plt.show()